# Wordle Çözücü Yazmak: Python ile Kelime Oyununu Yenmek

[![Colab'da Aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haticeozbolat01/pop-kultur-kodluyorum/blob/main/01-wordle-cozucu.ipynb)

Bu not defteri, haticeozbolat.com'daki **Popüler Kültürü Kodluyorum** serisinin ilk yazısına eşlik eder: [YAZI LİNKİ]

> **Önce kendi kopyanı al:** Üst menüden **Dosya > Drive'da kopya kaydet**'i seç.

Hücreleri sırayla **Shift + Enter** ile çalıştır.

## Hazırlık

Kelime sıklığı kütüphanesi `wordfreq`'i kuruyor ve açık lisanslı Türkçe yazım sözlüğünü ([hunspell-tr](https://github.com/tdd-ai/hunspell-tr), MPL-2.0) indiriyoruz. Sözlük dosyası yaklaşık 36 MB, inmesi birkaç saniye sürebilir.

In [ ]:
!pip install -q wordfreq
!wget -q -O tr_TR.dic https://raw.githubusercontent.com/tdd-ai/hunspell-tr/master/tr_TR.dic
print("Hazır")

## Çözücünün fonksiyonları

Yazıda adım adım anlattığımız dört fonksiyon: kelime listesini yüklemek, renkleri hesaplamak, listeyi daraltmak ve en iyi tahmini seçmek. Bir de çözücünün kendi kendine oynadığı `oyna` fonksiyonu.

In [ ]:
import sys
from collections import Counter

# ---------- 1. Kelime listesini hazırla ----------
TR_HARFLER = set("abcçdefgğhıijklmnoöprsştuüvyz")
# Listeden çıkardığımız birkaç argo kelime
CIKAR = {"kahpe", "sikik", "götün"}

def turkce_kucuk(metin):
    # Python'ın lower() fonksiyonu "I" harfini "i" yapar; Türkçede "ı" olmalı.
    return metin.replace("I", "ı").replace("İ", "i").lower()

def kelime_listesi_yukle(dic_yolu, en_fazla=60000):
    import wordfreq
    kokler = set()
    with open(dic_yolu, encoding="utf-8") as dosya:
        next(dosya)  # ilk satır kelime sayısı
        for satir in dosya:
            kelime = turkce_kucuk(satir.split("/")[0].strip())
            if len(kelime) == 5 and set(kelime) <= TR_HARFLER:
                kokler.add(kelime)
    # Sık kullanılan kelimelerle kesiştir: hem gerçek kelime hem tanıdık olsun
    sik = wordfreq.top_n_list("tr", en_fazla)
    return [k for k in sik if k in kokler and k not in CIKAR]

# ---------- 2. Oyunun kuralı: renkleri hesapla ----------
def renkleri_hesapla(tahmin, cevap):
    """Y = yeşil (doğru yer), S = sarı (yanlış yer), G = gri (yok)."""
    sonuc = ["G"] * 5
    kalan = Counter()
    # Önce yeşiller
    for i in range(5):
        if tahmin[i] == cevap[i]:
            sonuc[i] = "Y"
        else:
            kalan[cevap[i]] += 1
    # Sonra sarılar: aynı harf iki kez varsa fazladan sarı vermemek için sayaç kullanıyoruz
    for i in range(5):
        if sonuc[i] == "G" and kalan[tahmin[i]] > 0:
            sonuc[i] = "S"
            kalan[tahmin[i]] -= 1
    return "".join(sonuc)

# ---------- 3. Listeyi daralt ----------
def listeyi_daralt(adaylar, tahmin, renkler):
    # Cevap bu kelime olsaydı aynı renkleri görür müydük? Görürsek aday kalır.
    return [k for k in adaylar if renkleri_hesapla(tahmin, k) == renkler]

# ---------- 4. En iyi tahmini seç ----------
def en_iyi_tahmin(adaylar):
    # Her harfin kaç kelimede geçtiğini say (bir kelimede iki kez geçse de bir say)
    harf_sayisi = Counter()
    for k in adaylar:
        harf_sayisi.update(set(k))
    def puan(kelime):
        return sum(harf_sayisi[h] for h in set(kelime))
    return max(adaylar, key=puan)

# ---------- 5. Kendi kendine oyna ----------
def oyna(cevap, kelimeler, ilk_tahmin=None):
    adaylar = kelimeler
    for deneme in range(1, 7):
        tahmin = ilk_tahmin if (deneme == 1 and ilk_tahmin) else en_iyi_tahmin(adaylar)
        renkler = renkleri_hesapla(tahmin, cevap)
        if renkler == "YYYYY":
            return deneme
        adaylar = listeyi_daralt(adaylar, tahmin, renkler)
    return None  # 6 denemede bulamadı

## Kelime listesini yükle

In [ ]:
kelimeler = kelime_listesi_yukle("tr_TR.dic")
print("Kelime sayısı:", len(kelimeler))
print(kelimeler[:20])

## Renk fonksiyonunu dene

In [ ]:
print(renkleri_hesapla("elmas", "kalem"))  # SSSSG
print(renkleri_hesapla("kalem", "kalem"))  # YYYYY

## Bir oyunu adım adım izle

Cevabı istediğin kelimeyle değiştirip yeniden çalıştırabilirsin.

In [ ]:
cevap = "kalem"
adaylar = kelimeler
for deneme in range(1, 7):
    tahmin = en_iyi_tahmin(adaylar)
    renkler = renkleri_hesapla(tahmin, cevap)
    adaylar = listeyi_daralt(adaylar, tahmin, renkler)
    print(deneme, tahmin, renkler, "kalan aday:", len(adaylar))
    if renkler == "YYYYY":
        break

## Çözücüyü tüm kelimelerde test et

Her kelimeyi sırayla cevap yapıp çözücüye oynatıyoruz. Bu hücre 30 saniye kadar sürebilir.

In [ ]:
ilk = en_iyi_tahmin(kelimeler)
print("İlk tahmin:", ilk)
sonuclar = Counter(oyna(c, kelimeler, ilk) for c in kelimeler)
for k in [1, 2, 3, 4, 5, 6, None]:
    print("Bulamadı" if k is None else f"{k} tahmin", ":", sonuclar[k])
bulunan = sum(sonuclar[k] for k in range(1, 7))
print("Ortalama:", round(sum(k * sonuclar[k] for k in range(1, 7)) / bulunan, 2))

## Kendi oyununda kullan

Her tahminden sonra `tahmin` ve `renkler` satırlarını değiştirip hücreyi yeniden çalıştır. Yeni bir oyuna başlarken üstteki `adaylar = kelimeler` hücresini de çalıştır.

In [ ]:
adaylar = kelimeler

In [ ]:
tahmin = "ekran"
renkler = "SSGSG"   # oyunda gördüğün renkler: Y yeşil, S sarı, G gri

adaylar = listeyi_daralt(adaylar, tahmin, renkler)
print(len(adaylar), "aday kaldı:", adaylar[:15])
print("Önerim:", en_iyi_tahmin(adaylar))

## Meydan okuma

Çözücü 43 kelimeyi altı tahminde bulamıyor (örneğin "senin": seçin, selin, sevin, senin tuzağı). `en_iyi_tahmin` fonksiyonunu geliştirip bu sayıyı düşürebilir misin? Sonucunu blog yazısının altına yorum olarak yaz.

---
Kelime listesi kaynakları: [hunspell-tr](https://github.com/tdd-ai/hunspell-tr) (MPL-2.0) ve [wordfreq](https://pypi.org/project/wordfreq/).